**11/23/2025**

##Converting Data from Xena TCGA-GBM Cohort into spike trains and saving in csv files for SNNs

Datasets: Methylation27k (DNA methylation), Phenotype (Clinical Data + ground truths), HiSeqV2 (mRNA seq)


https://xenabrowser.net/datapages/?cohort=GDC%20TCGA%20Glioblastoma%20(GBM)&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443


Read the column names of the clinical data to find which one contains the subtypes

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

clin = pd.read_csv("TCGA.GBM.sampleMap_GBM_clinicalMatrix", sep="\t", index_col=0)
print(clin.columns.tolist())


Saving HiSeqV2 to HiSeqV2
Saving HumanMethylation27 to HumanMethylation27
Saving TCGA.GBM.sampleMap_GBM_clinicalMatrix to TCGA.GBM.sampleMap_GBM_clinicalMatrix
['CDE_DxAge', 'CDE_alk_chemoradiation_standard', 'CDE_chemo_adjuvant_alk', 'CDE_chemo_adjuvant_tmz', 'CDE_chemo_alk', 'CDE_chemo_alk_days', 'CDE_chemo_alk_long', 'CDE_chemo_tmz', 'CDE_chemo_tmz_days', 'CDE_chemo_tmz_long', 'CDE_missing', 'CDE_missingflag', 'CDE_previously_treated', 'CDE_radiation_adjuvant', 'CDE_radiation_adjuvant_standard', 'CDE_radiation_adjuvant_standard_probable', 'CDE_radiation_any', 'CDE_radiation_standard', 'CDE_radiation_standard_probable', 'CDE_sourcesite', 'CDE_survival_time', 'CDE_suspect', 'CDE_therapy', 'CDE_tmz_chemoradiation_standard', 'CDE_vital_status', 'G_CIMP_STATUS', 'GeneExp_Subtype', 'In_Cancer_Cell_Paper', '_INTEGRATION', '_PANCAN_CNA_PANCAN_K8', '_PANCAN_Cluster_Cluster_PANCAN', '_PANCAN_DNAMethyl_GBM', '_PANCAN_DNAMethyl_PANCAN', '_PANCAN_RPPA_PANCAN_K8', '_PANCAN_UNC_RNAseq_PANCAN_K16',

Load the three datasets and convert to spike trains

| Modality                  | Encoding scheme          |
| ---------------------- | -------------------- |
| `RNA Seq`          | Rate Encoding    |
| `Methylation`          | Phase (First-Spike) Encoding      |
| `Clinical`            | Temporal (Address) Encoding     |



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Set the total simulation time for the spike trains (T time steps)
T = 10

# ===============================
# SPIKE ENCODING FUNCTIONS
# ===============================

def rate_encode(data, T=10, seed=42):
    """
    Rate Encoding: Spike probability proportional to the feature value (0 to 1).
    Assumes data is already scaled, typically to a range like [-1, 1].
    The data is first mapped to [0, 1] for probability calculation.
    Output shape: (N_samples, T, N_features)
    """
    np.random.seed(seed)
    # Map from standardized range (e.g., -1 to 1) to a rate probability [0, 1]
    # Use max/min to ensure data is strictly within bounds
    min_val = data.min().min()
    max_val = data.max().max()

    # Simple min-max scaling to [0, 1] to get the spiking probability (rate)
    rates = (data - min_val) / (max_val - min_val + 1e-6)

    # Create spike trains: (N_samples, N_features, T)
    spike_trains = np.zeros((data.shape[0], data.shape[1], T), dtype=int)

    for t in range(T):
        # Generate random numbers for each feature/sample
        random_matrix = np.random.rand(*data.shape)
        # Spike if the random number is less than the feature's rate
        spike_trains[:, :, t] = (random_matrix < rates.values).astype(int)

    # Reshape to (N_samples, T, N_features) as is often preferred in SNN libraries (T dimension second)
    # Transpose is not strictly necessary but can be useful depending on SNN framework input requirements.
    # We will keep it as (N_samples, N_features, T) for easier saving/reshaping later.
    return spike_trains.reshape(data.shape[0], data.shape[1] * T) # Flatten T and N_features for saving

def phase_encode(data, T=10):
    """
    Phase Encoding (First-Spike Time): Feature value mapped to the time of the first spike.
    Assumes data is already scaled (e.g., standard normal).
    Higher value -> earlier spike time.
    Output shape: (N_samples, T, N_features)
    """
    # Map standardized data to a time range [0, T-1]
    # Map the min/max of the data to the latest/earliest time step.
    min_val = data.min().min()
    max_val = data.max().max()

    # Map range [min_val, max_val] to a time index [T-1, 0]
    normalized_data = (data.values - min_val) / (max_val - min_val + 1e-6)
    # Time index goes from T-1 (lowest value) to 0 (highest value)
    time_indices = np.floor((1 - normalized_data) * (T - 1)).astype(int)

    spike_trains = np.zeros((data.shape[0], data.shape[1], T), dtype=int)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            # Set a spike at the calculated time step
            spike_trains[i, j, time_indices[i, j]] = 1

    return spike_trains.reshape(data.shape[0], data.shape[1] * T)

def temporal_encode(data, T=10):
    """
    Temporal/Address Encoding: Each feature is assigned a specific time step.
    Suitable for sparse one-hot encoded categorical data.
    Requires T >= N_features (or T needs to be a multiple/combination).
    If T < N_features, features will be mapped to the same time step.
    Output shape: (N_samples, T, N_features)
    """
    N_features = data.shape[1]

    if T < N_features:
        print(f"Warning: T={T} is less than N_features={N_features}. Time steps will be reused.")

    spike_trains = np.zeros((data.shape[0], N_features, T), dtype=int)

    # Assign each feature to a specific time step (index)
    # Feature j is mapped to time step j % T
    for j in range(N_features):
        time_step = j % T
        # Only spike if the feature value is 1 (for one-hot data)
        spike_trains[:, j, time_step] = (data.iloc[:, j].values == 1).astype(int)

    return spike_trains.reshape(data.shape[0], N_features * T)

# -------------------------------
# LOAD RAW FILES
# -------------------------------
rna = pd.read_csv("HiSeqV2", sep="\t", index_col=0)
meth = pd.read_csv("HumanMethylation27", sep="\t", index_col=0)
clin = pd.read_csv("TCGA.GBM.sampleMap_GBM_clinicalMatrix", sep="\t", index_col=0)

# -------------------------------
# CLEAN SAMPLE IDS
# -------------------------------
def clean_barcode(x):
    return x.str[:12].str.upper()

rna.columns = clean_barcode(rna.columns)
meth.columns = clean_barcode(meth.columns)
clin.index = clean_barcode(clin.index)

# -------------------------------
# DROP EMPTY / CONSTANT CLINICAL COLUMNS
# -------------------------------
def drop_empty(df, threshold=0.95):
    keep = df.isna().mean() < threshold
    df = df.loc[:, keep]
    nunique = df.nunique()
    df = df.loc[:, nunique > 1]
    return df

clin = drop_empty(clin)

# -------------------------------
# EXTRACT LABEL COLUMN
# -------------------------------
label_candidates = ["GeneExp_Subtype", "Subtype_Classification", "TCGA_Subtype",
                    "PAM50", "GeneExpression.Subtype", "Subtype"]

subtype_col = next((c for c in label_candidates if c in clin.columns), None)
if subtype_col is None:
    raise ValueError("No subtype column found in clinical matrix.")

labels = clin[subtype_col].astype(str)
clin = clin.drop(columns=[subtype_col])

# -------------------------------
# INTERSECT SAMPLES
# -------------------------------
common_samples = sorted(list(
    set(rna.columns) & set(meth.columns) & set(clin.index) & set(labels.index)
))

rna = rna[common_samples].T
meth = meth[common_samples].T
clin = clin.loc[common_samples]
labels = labels.loc[common_samples]

# -------------------------------
# REMOVE DUPLICATES
# -------------------------------
def remove_dupes(df, name):
    if df.index.duplicated().any():
        print(f"Removing duplicates in {name}: {df.index[df.index.duplicated()].tolist()}")
    return df[~df.index.duplicated(keep="first")]

rna = remove_dupes(rna, "RNA")
meth = remove_dupes(meth, "Methylation")
clin = remove_dupes(clin, "Clinical")
labels = labels[~labels.index.duplicated(keep="first")]

# -------------------------------
# REPLACE NaNs / INFs
# -------------------------------
# Use -1 for NaNs/INFs to ensure they are handled as a low value during encoding
rna = rna.replace([np.inf, -np.inf], np.nan).fillna(-1).astype(float)
meth = meth.replace([np.inf, -np.inf], np.nan).fillna(-1).astype(float)
clin = clin.replace([np.inf, -np.inf], np.nan).fillna("-999") # fill NaNs in clinical categorical with a placeholder

# -------------------------------
# RNA PRE-ENCODING (top 1000 variable genes)
# -------------------------------
top_n_genes = 1000
gene_var = rna.var(axis=0)
top_genes = gene_var.sort_values(ascending=False).head(top_n_genes).index
rna_selected = rna[top_genes]

# Standardize RNA (required for rate encoding)
rna_standardized = pd.DataFrame(
    StandardScaler().fit_transform(rna_selected),
    index=rna_selected.index,
    columns=rna_selected.columns
)
# Perform Rate Encoding
rna_spikes_array = rate_encode(rna_standardized, T=T)
rna_spikes = pd.DataFrame(
    rna_spikes_array,
    index=rna_standardized.index,
    columns=[f"RNA_{c}_t{t}" for c in rna_standardized.columns for t in range(T)]
)
print("RNA Spike Trains Shape:", rna_spikes.shape)

# -------------------------------
# METHYLATION PRE-ENCODING (Top DMPs - FIXED LEAKAGE)
# -------------------------------
from sklearn.feature_selection import SelectKBest

# 1. Clip and align data
meth_clipped = meth.clip(0, 1)
common_DMP_samples = meth_clipped.index.intersection(labels.index)

meth_DMP = meth_clipped.loc[common_DMP_samples]

# 2. Select top K probes based on VARIANCE (Label-Agnostic)
top_n_probes = 5000

# Calculate variance for each probe
probe_var = meth_DMP.var(axis=0)

# Select the top N probes by variance
top_probes = probe_var.sort_values(ascending=False).head(top_n_probes).index

meth_selected = meth_DMP[top_probes]
print(f"Methylation probes selected: {meth_selected.shape[1]}")


# 3. Standardize and Encode
meth_standardized = pd.DataFrame(
    StandardScaler().fit_transform(meth_selected),
    index=meth_selected.index,
    columns=meth_selected.columns
)
# Perform Phase Encoding
meth_spikes_array = phase_encode(meth_standardized, T=T)
meth_spikes = pd.DataFrame(
    meth_spikes_array,
    index=meth_standardized.index,
    columns=[f"METH_{c}_t{t}" for c in meth_standardized.columns for t in range(T)]
)
print("Methylation Spike Trains Shape:", meth_spikes.shape)

# -------------------------------
# CLINICAL PRE-ENCODING
# -------------------------------
numeric_cols = clin.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = clin.select_dtypes(exclude=[np.number]).columns.tolist()

# Numeric data is kept separate or could be rate/phase encoded,
# but for simplicity for SNNs, we will drop non-categorical numeric
# or convert to one-hot if it represents limited categories.
# For this example, we keep only one-hot encoded categorical data.
clin_cat = pd.get_dummies(clin[cat_cols], drop_first=True)

# Perform Temporal Encoding on One-Hot Clinical data
clin_spikes_array = temporal_encode(clin_cat, T=T)
clin_spikes = pd.DataFrame(
    clin_spikes_array,
    index=clin_cat.index,
    columns=[f"CLIN_{c}_t{t}" for c in clin_cat.columns for t in range(T)]
)
print("Clinical Spike Trains Shape:", clin_spikes.shape)


# -------------------------------
# ENCODE LABELS
# -------------------------------
le = LabelEncoder()
labels_num = pd.Series(le.fit_transform(labels), index=labels.index, name="label")
print("Label distribution:\n", labels_num.value_counts())
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# -------------------------------
# FINAL ALIGNMENT BEFORE CONCAT
# -------------------------------
common_final = rna_spikes.index.intersection(meth_spikes.index).intersection(clin_spikes.index).intersection(labels_num.index)
rna_spikes = rna_spikes.loc[common_final]
meth_spikes = meth_spikes.loc[common_final]
clin_spikes = clin_spikes.loc[common_final]
labels_num = labels_num.loc[common_final]

# -------------------------------
# COMBINE MODALITIES
# -------------------------------
# X is now the combined spike trains
X_spikes = pd.concat([rna_spikes, meth_spikes, clin_spikes], axis=1)

# -------------------------------
# SAVE DATA
# -------------------------------
X_spikes.to_csv("features_spiketrains.csv")
labels_num.to_csv("labels_numeric.csv")
print("\nSaved: features_spiketrains.csv, labels_numeric.csv")
print("Final spike train matrix shape:", X_spikes.shape)

Removing duplicates in RNA: ['TCGA-06-0125', 'TCGA-14-1034']
Removing duplicates in Clinical: ['TCGA-06-0125', 'TCGA-14-0736', 'TCGA-14-1034', 'TCGA-14-1402', 'TCGA-19-0957', 'TCGA-19-1389']
RNA Spike Trains Shape: (80, 10000)


/tmp/ipython-input-3283822558.py:165: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clin = clin.replace([np.inf, -np.inf], np.nan).fillna("-999") # fill NaNs in clinical categorical with a placeholder


Methylation probes selected: 5000
Methylation Spike Trains Shape: (80, 50000)
Clinical Spike Trains Shape: (80, 25510)
Label distribution:
 label
1    29
0    21
3    18
2    11
4     1
Name: count, dtype: int64
Label mapping: {'Classical': np.int64(0), 'Mesenchymal': np.int64(1), 'Neural': np.int64(2), 'Proneural': np.int64(3), 'nan': np.int64(4)}

Saved: features_spiketrains.csv, labels_numeric.csv
Final spike train matrix shape: (80, 85510)


Now save the compiled data for SNNs

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------
# LOAD SPIKE TRAIN DATA
# -------------------------------
X = pd.read_csv("features_spiketrains.csv", index_col=0)
y = pd.read_csv("labels_numeric.csv", index_col=0).iloc[:, 0]

print("Loaded spike train matrix:", X.shape)
print("Loaded labels:", y.shape)

# -------------------------------
# ALIGN & VERIFY SAMPLE CONSISTENCY
# -------------------------------
common = sorted(list(set(X.index) & set(y.index)))
X = X.loc[common]
y = y.loc[common]

print("Samples after alignment:", len(common))

# -------------------------------
# REMOVE RARE CLASSES
# -------------------------------
min_samples_per_class = 2
counts = y.value_counts()
rare_classes = counts[counts < min_samples_per_class].index.tolist()
if rare_classes:
    print("Removing rare classes:", rare_classes)
    mask = ~y.isin(rare_classes)
    X = X.loc[mask]
    y = y.loc[mask]

print("Label distribution after removal:\n", y.value_counts())

# -------------------------------
# STRATIFIED TRAIN / VAL / TEST SPLIT
# 60% train, 20% val, 20% test
# -------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# -------------------------------
# PRINT SPLIT SIZES
# -------------------------------
print("\nFINAL SPLIT SIZES:")
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))
print("Total:", len(X_train) + len(X_val) + len(X_test))

# -------------------------------
# SAVE SPLITS
# -------------------------------
X_train.to_csv("X_train_spikes.csv")
y_train.to_csv("y_train_spikes.csv") # Renaming y files for consistency, though they remain numeric

X_val.to_csv("X_val_spikes.csv")
y_val.to_csv("y_val_spikes.csv")

X_test.to_csv("X_test_spikes.csv")
y_test.to_csv("y_test_spikes.csv")

print("\nSaved:")
print("   X_train_spikes.csv, y_train_spikes.csv")
print("   X_val_spikes.csv, y_val_spikes.csv")
print("   X_test_spikes.csv, y_test_spikes.csv")

Loaded spike train matrix: (80, 85510)
Loaded labels: (80,)
Samples after alignment: 80
Removing rare classes: [4]
Label distribution after removal:
 label
1    29
0    21
3    18
2    11
Name: count, dtype: int64

FINAL SPLIT SIZES:
Train: 47
Validation: 16
Test: 16
Total: 79

Saved:
   X_train_spikes.csv, y_train_spikes.csv
   X_val_spikes.csv, y_val_spikes.csv
   X_test_spikes.csv, y_test_spikes.csv


Export .csv files:

| File                   | Description          |
| ---------------------- | -------------------- |
| `X_train_spikes.csv`          | training features    |
| `y_train_spikes.csv`          | training labels      |
| `X_val_spikes.csv`            | cross-validation     |
| `y_val_spikes.csv`           | validation labels     |
| `X_test_spikes.csv` | final test data |
| `y_test_spikes.csv`   | final test labels  |

Will manually save to Google Drive (Folder = `Multimodal_SNN_Protocol_Project`) and Github


In [ ]:
from google.colab import files

# ---- SAVE ----
X_train.to_csv("X_train_spikes.csv", index=True)
y_train.to_csv("y_train_spikes.csv", index=True)

X_val.to_csv("X_val_spikes.csv", index=True)
y_val.to_csv("y_val_spikes.csv", index=True)

X_test.to_csv("X_test_spikes.csv", index=True)
y_test.to_csv("y_test_spikes.csv", index=True)


print("Files saved. Starting downloads...")

# ---- DOWNLOAD ----
files.download("X_train_spikes.csv")
files.download("y_train_spikes.csv")

files.download("X_val_spikes.csv")
files.download("y_val_spikes.csv")

files.download("X_test_spikes.csv")
files.download("y_test_spikes.csv")

Files saved. Starting downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>